# IndicOCR on Google Colab

Before running:
1. **Runtime > Change runtime type > T4 GPU** (or better).
2. Accept the license at https://huggingface.co/bodhan-ai/indic-ocr
3. Add your Hugging Face token: click the **key icon (Secrets)** on the left, add a secret named `HF_TOKEN`, paste your token, and turn on **Notebook access**.

Then run the cells top to bottom.

In [ ]:
# 1. Check the GPU
!nvidia-smi

In [ ]:
# 2. Install EVERYTHING once, before anything is imported (installing later crashes the session)
#    pillow<13 because gradio does not support pillow 13 yet
!pip install -q -U "transformers>=5.7" "accelerate>=1.1" "huggingface_hub>=1.0" "pillow>=10.0.1,<13"     ultralytics gradio openpyxl lxml
print("Done. Now: Runtime > Restart session, then continue from cell 3 (do not re-run this cell).")

In [ ]:
# 3. Log in to Hugging Face and download the model + weights (~1.8 GB)
from google.colab import userdata
from huggingface_hub import login, snapshot_download

login(userdata.get("HF_TOKEN"))
repo = snapshot_download("bodhan-ai/indic-ocr", local_dir="/content/indic-ocr")
print("Downloaded to", repo)

In [ ]:
# 4. Load the model (once per session)
import sys, torch
sys.path.insert(0, repo)
from indic_ocr import IndicOCR

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
parser = IndicOCR.from_pretrained(repo)   # add table_format="markdown" for Markdown tables

In [ ]:
# 4b. Speed fix: run the text reader in fp16 (the T4 has no fast bf16). ~5x faster, same accuracy.
import gc, torch
parser = IndicOCR.from_pretrained(repo)
parser._rec_cfg = parser._rec_cfg.merged(dtype="float16")
gc.collect(); torch.cuda.empty_cache()
_ = parser.recognizer   # load it now
print("fp16 parser ready:", parser._rec_cfg.dtype)

In [ ]:
# 5. Upload page images (PNG/JPG) from your computer
from google.colab import files
uploaded = files.upload()
images = list(uploaded.keys())
images

In [ ]:
# 6. Run OCR, print the text, and save Markdown + JSON for each image
import json, os

def fix_spaces(s):
    # the tokenizer sometimes leaves its space marker "▁" in the decoded text
    return s.replace("▁", " ") if s else s

os.makedirs("/content/output", exist_ok=True)
pages = {}
for img in images:
    page = parser.parse(img)
    page["markdown"] = fix_spaces(page["markdown"])
    for b in page["blocks"]:
        b["text"] = fix_spaces(b.get("text", ""))
    pages[img] = page

    name = os.path.splitext(img)[0]
    with open(f"/content/output/{name}.md", "w", encoding="utf-8") as f:
        f.write(page["markdown"])
    with open(f"/content/output/{name}.json", "w", encoding="utf-8") as f:
        json.dump(page, f, ensure_ascii=False, indent=2)
    print(f"===== {img} =====")
    print(page["markdown"])

In [ ]:
# 7. Visualize: numbered, colour-coded blocks (like the IndicOCR diagram) + rendered Markdown
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display, Markdown

COLORS = {
    "Title": "#F97316", "SectionHeader": "#F97316", "Text": "#3B82F6", "Table": "#16A34A",
    "Equation": "#8B5CF6", "Caption": "#0EA5E9", "Footnote": "#64748B",
    "PageHeader": "#94A3B8", "PageFooter": "#94A3B8", "PageNumber": "#94A3B8",
    "Figure": "#EC4899", "Picture": "#EC4899",
}
import glob, json, os
if "pages" not in globals():
    # Cell 6 was not run in this session: rebuild results from the saved JSON files
    pages = {}
    for jf in glob.glob("/content/output/*.json"):
        page = json.load(open(jf, encoding="utf-8"))
        for b in page["blocks"]:
            b["text"] = (b.get("text") or "").replace("▁", " ")
        page["markdown"] = (page.get("markdown") or "").replace("▁", " ")
        pages[page["image"]] = page
    print("loaded", len(pages), "page(s) from /content/output")

font = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", 22)

for img, page in pages.items():
    im = Image.open(img).convert("RGB")
    overlay = Image.new("RGBA", im.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(overlay)
    for b in page["blocks"]:
        x0, y0, x1, y1 = b["bbox_xyxy"]
        c = COLORS.get(b["type"], "#3B82F6")
        rgb = tuple(int(c[i:i + 2], 16) for i in (1, 3, 5))
        d.rectangle([x0, y0, x1, y1], fill=rgb + (40,), outline=rgb + (255,), width=3)
        tag = f"{b['order'] + 1} {b['label']}"
        tw = d.textlength(tag, font=font)
        d.rectangle([x0, y0 - 28, x0 + tw + 10, y0], fill=rgb + (255,))
        d.text((x0 + 5, y0 - 26), tag, fill="white", font=font)
    out = Image.alpha_composite(im.convert("RGBA"), overlay).convert("RGB")
    name = os.path.splitext(img)[0]
    out.save(f"/content/output/{name}_layout.png")
    display(out.resize((out.width * 900 // out.height, 900)) if out.height > 900 else out)
    display(Markdown(page["markdown"]))

In [ ]:
# 8. Download all results (.md, .json, _layout.png) as a zip
!cd /content && rm -f output.zip && zip -qr output.zip output
files.download("/content/output.zip")

# Benchmark: IndicOCR CER / WER on the AksharDrishti 24-page benchmark

Scores IndicOCR with **your own** `eval/metrics.py` (same `normalize_for_scoring`, corpus-level CER/WER),
so the numbers can sit as a 4th column next to Table 9.

Needs: Cells 1-4 above (model loaded), and the 24 benchmark page images
(`Doc001.png`, `doc_0001.png`, ... as named in `benchmark/metadata.csv`) in one folder, e.g. on Drive.

In [ ]:
# B1. Get the AksharDrishti repo (ground truth + metric code) and point to the benchmark images
!git clone -q --depth 1 https://github.com/Ayush-04-spec/akshardrishti /content/akshardrishti 2>/dev/null || echo "repo already cloned"

from google.colab import drive
drive.mount("/content/drive")

# >>> EDIT THIS: folder containing Doc001.png ... doc_0095.png <<<
BENCH_IMAGES = "/content/drive/MyDrive/benchmark_images"

In [ ]:
# B2. Run IndicOCR on every benchmark page
import csv, html, os, re, sys, json
import idp_contract

# AksharDrishti's ground truth includes headers/footers, but IndicOCR skips them by default.
# Read them too, so missing header text is not counted as an error that is only a config choice.
idp_contract.OCR_SKIP_LABELS = frozenset({"diagram", "image", "chart", "advertisement"})

def to_plain(text):
    """IndicOCR block text -> plain text comparable with gt_text (tables come back as HTML)."""
    t = (text or "").replace("▁", " ")
    t = re.sub(r"<br\s*/?>", " ", t)
    t = re.sub(r"</t[dh]>", " ", t)
    t = re.sub(r"</tr>", "\n", t)
    t = re.sub(r"<[^>]+>", "", t)
    t = html.unescape(t)
    t = t.replace("$", "")
    return t.strip()

meta = list(csv.DictReader(open("/content/akshardrishti/benchmark/metadata.csv", encoding="utf-8-sig")))
os.makedirs("/content/bench_out", exist_ok=True)
refs, hyps, rows = [], [], []
for m in meta:
    img = os.path.join(BENCH_IMAGES, m["original_filename"])
    gt_path = f"/content/akshardrishti/benchmark/gt_text/{m['line_id']}.txt"
    if not os.path.exists(img):
        print("MISSING image:", m["original_filename"], "-> skipped")
        continue
    page = parser.parse(img)
    hyp = "\n".join(to_plain(b.get("text")) for b in page["blocks"] if b.get("text"))
    ref = open(gt_path, encoding="utf-8").read()
    open(f"/content/bench_out/{m['line_id']}.txt", "w", encoding="utf-8").write(hyp)
    refs.append(ref); hyps.append(hyp); rows.append(m)
    print("done", m["line_id"], m["original_filename"])
print(len(rows), "pages scored")

In [ ]:
# B3. CER / WER with AksharDrishti's own metric code
sys.path.insert(0, "/content/akshardrishti/eval")
sys.path.insert(0, "/content/akshardrishti")
from metrics import corpus_cer, corpus_wer, cer, wer, bootstrap_ci

c = corpus_cer(refs, hyps)
w = corpus_wer(refs, hyps)
lo, hi = bootstrap_ci(c["per_document_cer"])
print(f"IndicOCR on {c['n_documents']} pages")
print(f"  CER = {c['cer']:.3f}   (95% CI {lo:.3f}-{hi:.3f}, mean per-doc CER {c['mean_document_cer']:.3f})")
print(f"  WER = {w['wer']:.3f}")
print()
print("Compare (paper Table 9): Layout ON CER 0.080 / WER 0.089 | Layout OFF 0.050 / 0.079 | Tesseract 0.108 / 0.127")
print()
print(f"{'page':8} {'scan':15} {'CER':>6} {'WER':>6}")
for m, r, h in zip(rows, refs, hyps):
    print(f"{m['line_id']:8} {m['scan_type']:15} {cer(r, h):6.3f} {wer(r, h):6.3f}")

json.dump({"cer": c, "wer": w, "ci95": [lo, hi]}, open("/content/bench_out/indicocr_results.json", "w"), indent=2)

# Stamps & signatures: YOLO + IndicOCR

IndicOCR has no stamp/seal or signature class, so these are detected separately and merged into the same JSON.

- **Signatures**: pretrained YOLOv8s from `tech4humans/yolov8s-signature-detector` (accept its terms on Hugging Face first if asked).
- **Stamps**: colour-ink detector (blue/red/violet seals), no training needed. If you later train your own stamp YOLO, set `STAMP_WEIGHTS` and it is used instead.

Needs: Cells 1-6 above (model loaded, `images` uploaded, `pages` filled).

In [ ]:
# S1. Install YOLO and load the detectors
# (ultralytics was installed in cell 2)
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

sig_model = YOLO(hf_hub_download("tech4humans/yolov8s-signature-detector", "yolov8s.pt"))

STAMP_WEIGHTS = None   # e.g. "/content/drive/MyDrive/stamp_best.pt" once you have a trained stamp model
stamp_model = YOLO(STAMP_WEIGHTS) if STAMP_WEIGHTS else None
print("signature classes:", sig_model.names, "| stamp model:", STAMP_WEIGHTS or "colour detector")

In [ ]:
# S2. Detection + merge helpers
import cv2, numpy as np, os, json

def covered(a, b):
    """Fraction of box a that lies inside box b."""
    x0, y0, x1, y1 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, x1 - x0) * max(0, y1 - y0)
    area = (a[2] - a[0]) * (a[3] - a[1])
    return inter / area if area else 0.0

def yolo_boxes(model, path, kind, conf):
    r = model(path, conf=conf, imgsz=1024, verbose=False)[0]
    return [{"type": kind, "bbox_xyxy": [round(v, 1) for v in b], "conf": round(c, 3), "method": "yolo"}
            for b, c in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist())]

def colour_stamps(path, min_area_frac=0.002):
    """Stamps are usually coloured ink: find large saturated blobs."""
    img = cv2.imread(path)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, (0, 70, 50), (180, 255, 255))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((25, 25), np.uint8))
    n, _, stats, _ = cv2.connectedComponentsWithStats(mask)
    H, W = mask.shape
    out = []
    for x, y, w, h, a in stats[1:]:
        if a < min_area_frac * H * W or not (0.4 < w / h < 2.5):
            continue
        out.append({"type": "stamp", "bbox_xyxy": [float(x), float(y), float(x + w), float(y + h)],
                    "conf": None, "method": "colour"})
    return out

def detect_marks(path, sig_conf=0.4, stamp_conf=0.4):
    sigs = yolo_boxes(sig_model, path, "signature", sig_conf)
    if stamp_model:
        stamps = yolo_boxes(stamp_model, path, "stamp", stamp_conf)
    else:
        # blue-ink signatures also look like colour blobs: drop blobs that are mostly a signature
        stamps = [s for s in colour_stamps(path)
                  if not any(covered(s["bbox_xyxy"], g["bbox_xyxy"]) > 0.6 for g in sigs)]
    return sigs + stamps

def add_marks(path, page, crop_dir="/content/output/crops"):
    from PIL import Image
    os.makedirs(crop_dir, exist_ok=True)
    im = Image.open(path).convert("RGB")
    marks = detect_marks(path)
    name = os.path.splitext(os.path.basename(path))[0]
    for i, m in enumerate(marks):
        m["crop"] = f"{crop_dir}/{name}_{m['type']}_{i}.png"
        im.crop(m["bbox_xyxy"]).save(m["crop"])
    # tag IndicOCR blocks that are really a stamp/signature (e.g. a seal read as a paragraph)
    for b in page["blocks"]:
        for m in marks:
            if covered(b["bbox_xyxy"], m["bbox_xyxy"]) > 0.5:
                b["inside_mark"] = m["type"]
    page["marks"] = marks
    return page

In [ ]:
# S3. Run on every page: save merged JSON and a picture with the stamp/signature boxes
from PIL import Image, ImageDraw
from IPython.display import display

MARK_COLORS = {"signature": (220, 38, 38), "stamp": (124, 58, 237)}
for img, page in pages.items():
    page = add_marks(img, page)
    name = os.path.splitext(img)[0]
    json.dump(page, open(f"/content/output/{name}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

    im = Image.open(img).convert("RGB")
    d = ImageDraw.Draw(im)
    for m in page["marks"]:
        c = MARK_COLORS[m["type"]]
        d.rectangle(m["bbox_xyxy"], outline=c, width=6)
        d.text((m["bbox_xyxy"][0] + 4, m["bbox_xyxy"][1] + 4), m["type"], fill=c)
    im.save(f"/content/output/{name}_marks.png")
    n_sig = sum(m["type"] == "signature" for m in page["marks"])
    n_stp = sum(m["type"] == "stamp" for m in page["marks"])
    n_tag = sum("inside_mark" in b for b in page["blocks"])
    print(f"{img}: {n_sig} signature(s), {n_stp} stamp(s), {n_tag} OCR block(s) sitting on a mark")
    display(im.resize((im.width * 900 // im.height, 900)) if im.height > 900 else im)

# then re-run Cell 8 to download everything (JSON now includes "marks", crops are in output/crops)

# AksharDrishti web app
Run order: cell 1, cell 2, **Runtime > Restart session**, then cells 3, 4, 4b, S1, S2, A2, A3, A4 and **W1**. W1 downloads the website files from GitHub and prints the link. A5 is the old app and is switched off.

In [ ]:
# A1. Check the app libraries (installed in cell 2)
import gradio, openpyxl, PIL
print("gradio", gradio.__version__, "| pillow", PIL.__version__)

In [ ]:
# A2. Key-field extraction + privacy masking
import re, io, os, html, json, base64, sqlite3, datetime, unicodedata
import pandas as pd
from PIL import Image, ImageDraw

DEV_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")   # Devanagari -> ASCII digits (1:1, so spans line up)

def clean(s):
    return (s or "").replace("▁", " ")

def to_plain(text):
    """Block text -> plain text (tables come back as HTML)."""
    t = clean(text)
    t = re.sub(r"<br\s*/?>", " ", t)
    t = re.sub(r"</t[dh]>", " ", t)
    t = re.sub(r"</tr>", "\n", t)
    t = re.sub(r"<[^>]+>", "", t)
    return html.unescape(t).replace("$", "").strip()

FIELD_PATTERNS = {
    "Aadhaar":          (r"(?<!\d)[2-9]\d{3}\s?\d{4}\s?\d{4}(?!\d)", 0),
    "PAN":              (r"(?<![A-Z0-9])[A-Z]{5}\d{4}[A-Z](?![A-Z0-9])", 0),
    "Phone":            (r"(?<!\d)(?:\+91[\s-]?)?[6-9]\d{4}\s?\d{5}(?!\d)", 0),
    "PIN code":         (r"(?<!\d)[1-9]\d{2}\s?\d{3}(?!\d)", 0),
    "Date":             (r"(?<!\d)\d{1,2}[./-]\d{1,2}[./-](?:\d{4}|\d{2})(?!\d)", 0),
    "Amount":           (r"(?:₹|rs\.?|रु\.?|रुपये)\s?\d[\d,]*(?:\.\d{1,2})?", re.I),
    "Survey / Gat no.": (r"(?:survey|sy\.|gat|गट|सर्वे|सर्व्हे)\s*(?:no\.?|number|नं\.?|क्र\.?|क्रमांक)?\s*[:.\-]?\s*\d+(?:/\d+)*", re.I),
    "Email":            (r"[\w.+-]+@[\w-]+\.[\w.]+", 0),
}
SENSITIVE = ("Aadhaar", "PAN")

def find_spans(text, kinds=None):
    """[(kind, start, end)] found in text (Devanagari digits included)."""
    t = (text or "").translate(DEV_DIGITS)
    out = []
    for kind, (pat, flags) in FIELD_PATTERNS.items():
        if kinds and kind not in kinds:
            continue
        out += [(kind, m.start(), m.end()) for m in re.finditer(pat, t, flags)]
    return out

def extract_fields(text):
    t = (text or "").translate(DEV_DIGITS)
    rows, seen = [], set()
    for kind, s, e in find_spans(text):
        v = " ".join(t[s:e].split())
        if (kind, v) not in seen:
            seen.add((kind, v)); rows.append([kind, v])
    return rows

def mask_value(kind, v):
    d = re.sub(r"\s", "", v.translate(DEV_DIGITS))
    return "XXXX XXXX " + d[-4:] if kind == "Aadhaar" else "XXXXXX" + d[-4:]

def mask_text(text):
    if not text:
        return text
    for kind, s, e in sorted(find_spans(text, SENSITIVE), key=lambda x: -x[1]):
        text = text[:s] + mask_value(kind, text[s:e]) + text[e:]
    return text

def sensitive_boxes(page):
    """Approximate image boxes of Aadhaar/PAN: the matching line band inside the block."""
    boxes = []
    for b in page["blocks"]:
        lines = to_plain(b.get("text")).split("\n")
        x0, y0, x1, y1 = b["bbox_xyxy"]
        lh = (y1 - y0) / max(len(lines), 1)
        for i, line in enumerate(lines):
            for _, s, e in find_spans(line, SENSITIVE):
                n = max(len(line), 1)
                bx0 = x0 + (x1 - x0) * max(s / n - 0.05, 0)
                bx1 = x0 + (x1 - x0) * min(e / n + 0.05, 1)
                boxes.append([bx0, y0 + i * lh, bx1, y0 + (i + 1) * lh])
    return boxes

# quick self-test
_t = "आधार क्र. २३४५ ६७८९ ०१२३, PAN ABCDE1234F, mob 9876543210, dt 12/03/2024, गट नं. 45/2, ₹ 1,500"
print(extract_fields(_t)); print(mask_text(_t))

In [ ]:
# A3. Searchable archive (SQLite) - every processed document is stored here
DB = "/content/archive.db"          # change to a Drive path to keep it between sessions
con = sqlite3.connect(DB, check_same_thread=False)
con.execute("""CREATE TABLE IF NOT EXISTS docs(
    id INTEGER PRIMARY KEY, name TEXT, added TEXT,
    signed INT, stamped INT, fields TEXT, text TEXT)""")
con.commit()

def nfc(s):
    return unicodedata.normalize("NFC", s or "")

def archive_add(name, signed, stamped, fields, text):
    con.execute("INSERT INTO docs(name, added, signed, stamped, fields, text) VALUES (?,?,?,?,?,?)",
                (name, datetime.datetime.now().strftime("%Y-%m-%d %H:%M"), int(signed), int(stamped),
                 json.dumps(fields, ensure_ascii=False), nfc(text)))
    con.commit()

def archive_search(q):
    q = nfc(q).strip()
    rows = con.execute("SELECT id, name, added, signed, stamped, text FROM docs WHERE text LIKE ? OR name LIKE ? "
                       "ORDER BY id DESC", (f"%{q}%", f"%{q}%")).fetchall()
    out = []
    for i, name, added, sg, st, text in rows:
        k = text.find(q) if q else 0
        snip = text[max(k - 60, 0): k + len(q) + 60].replace("\n", " ") if k >= 0 else text[:120]
        out.append([i, name, added, "✔" if sg else "✘", "✔" if st else "✘", "…" + snip + "…"])
    return pd.DataFrame(out, columns=["ID", "Document", "Added", "Signed", "Stamped", "Match"])

In [ ]:
# A4. Full pipeline: OCR -> stamps/signatures -> fields -> masking -> exports -> archive
BLOCK_COLORS = {"Table": (22, 163, 74), "Title": (249, 115, 22), "SectionHeader": (249, 115, 22)}
MARK_COLORS = {"signature": (220, 38, 38), "stamp": (124, 58, 237)}
MAX_SIDE = 2000
import gc, torch, time

def report_html(name, verdict, fields, page, img_path):
    b64 = base64.b64encode(open(img_path, "rb").read()).decode()
    rows = "".join(f"<tr><td>{html.escape(k)}</td><td>{html.escape(v)}</td></tr>" for k, v in fields)
    body = ""
    for b in sorted(page["blocks"], key=lambda b: b["order"]):
        t = clean(b.get("text"))
        if not t:
            continue
        body += t if b.get("type") == "Table" else f"<p>{html.escape(t)}</p>"
    return f"""<!DOCTYPE html><html><head><meta charset="utf-8"><title>{html.escape(name)}</title>
<style>body{{font-family:'Noto Sans','Noto Sans Devanagari',sans-serif;max-width:900px;margin:auto;padding:20px}}
table{{border-collapse:collapse;margin:10px 0}}td,th{{border:1px solid #999;padding:4px 8px}}img{{max-width:100%}}</style>
</head><body><h1>Document report: {html.escape(name)}</h1>
<p>Generated {datetime.datetime.now():%Y-%m-%d %H:%M} by AksharDrishti (OCR engine: IndicOCR by Bodhan AI)</p>
<h2>Verification</h2><pre>{html.escape(verdict)}</pre>
<h2>Key fields</h2><table><tr><th>Field</th><th>Value</th></tr>{rows}</table>
<h2>Annotated page</h2><img src="data:image/png;base64,{b64}">
<h2>Extracted text</h2>{body}</body></html>"""

def process_document(path, mask=True):
    if not path:
        raise gr.Error("Upload a page image first.")
    name = os.path.splitext(os.path.basename(path))[0]
    out = f"/content/app_output/{name}"
    os.makedirs(out, exist_ok=True)

    # 0. shrink big scans/photos (they crash Colab's RAM); 2000 px long side is plenty for OCR
    im0 = Image.open(path).convert("RGB")
    if max(im0.size) > MAX_SIDE:
        im0.thumbnail((MAX_SIDE, MAX_SIDE))
    path = f"{out}/{name}_input.png"
    im0.save(path)
    del im0

    # 1. OCR (IndicOCR)
    t0 = time.time()
    page = parser.parse(path)
    t1 = time.time()
    page["markdown"] = clean(page["markdown"])
    for b in page["blocks"]:
        b["text"] = clean(b.get("text"))

    # 2. stamps + signatures (cells S1/S2)
    page = add_marks(path, page, crop_dir=f"{out}/crops")
    t2 = time.time()
    sigs = [m for m in page["marks"] if m["type"] == "signature"]
    stamps = [m for m in page["marks"] if m["type"] == "stamp"]

    # 3. key fields
    full = "\n".join(to_plain(b["text"]) for b in sorted(page["blocks"], key=lambda b: b["order"]) if b["text"])
    fields = extract_fields(full)

    # 4. privacy masking
    hide = sensitive_boxes(page) if mask else []
    if mask:
        fields = [[k, mask_value(k, v) if k in SENSITIVE else v] for k, v in fields]
        for b in page["blocks"]:
            b["text"] = mask_text(b["text"])
        page["markdown"] = mask_text(page["markdown"])
        full = mask_text(full)
        for m in page["marks"]:   # crops could show a number next to a signature
            im_c = Image.open(m["crop"])
            dx, dy = m["bbox_xyxy"][0], m["bbox_xyxy"][1]
            for hb in hide:
                ImageDraw.Draw(im_c).rectangle([hb[0] - dx, hb[1] - dy, hb[2] - dx, hb[3] - dy], fill="black")
            im_c.save(m["crop"])
    page["fields"] = [{"field": k, "value": v} for k, v in fields]

    # 5. verification verdict
    verdict = (f"Signature: {'✔ found (' + str(len(sigs)) + ')' if sigs else '✘ not found'}\n"
               f"Stamp/seal: {'✔ found (' + str(len(stamps)) + ')' if stamps else '✘ not found'}\n"
               f"Status: {'COMPLETE - signed and stamped' if sigs and stamps else 'INCOMPLETE - check manually'}")
    page["verification"] = {"signed": bool(sigs), "stamped": bool(stamps)}

    # 6. annotated image
    im = Image.open(path).convert("RGB")
    d = ImageDraw.Draw(im)
    for b in page["blocks"]:
        d.rectangle(b["bbox_xyxy"], outline=BLOCK_COLORS.get(b.get("type"), (59, 130, 246)), width=2)
    for m in page["marks"]:
        d.rectangle(m["bbox_xyxy"], outline=MARK_COLORS[m["type"]], width=6)
        d.text((m["bbox_xyxy"][0] + 4, m["bbox_xyxy"][1] + 4), m["type"].upper(), fill=MARK_COLORS[m["type"]])
    for hb in hide:
        d.rectangle(hb, fill="black")
    img_out = f"{out}/{name}_annotated.png"
    im.save(img_out)

    # 7. exports
    json_out, md_out, xlsx_out, html_out = (f"{out}/{name}.json", f"{out}/{name}.md",
                                            f"{out}/{name}.xlsx", f"{out}/{name}_report.html")
    json.dump(page, open(json_out, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    open(md_out, "w", encoding="utf-8").write(page["markdown"])
    tables = []
    for b in page["blocks"]:
        if b.get("type") == "Table" and "<table" in b["text"]:
            try:
                tables += pd.read_html(io.StringIO(b["text"]))
            except Exception as e:
                print("table not converted:", e)
    with pd.ExcelWriter(xlsx_out) as w:
        pd.DataFrame(fields, columns=["Field", "Value"]).to_excel(w, sheet_name="Fields", index=False)
        pd.DataFrame([{k: str(m[k]) for k in ("type", "bbox_xyxy", "conf", "method")} for m in page["marks"]],
                     columns=["type", "bbox_xyxy", "conf", "method"]).to_excel(w, sheet_name="Stamps_Signatures", index=False)
        for i, t in enumerate(tables, 1):
            t.to_excel(w, sheet_name=f"Table_{i}", index=False)
    open(html_out, "w", encoding="utf-8").write(report_html(name, verdict, fields, page, img_out))

    # 8. archive
    archive_add(name, bool(sigs), bool(stamps), fields, full)

    global LAST_TIMING
    LAST_TIMING = {"parser": t1 - t0, "marks": t2 - t1, "rest": time.time() - t2}
    del im, d
    gc.collect(); torch.cuda.empty_cache()
    return (verdict, img_out, page["markdown"], pd.DataFrame(fields, columns=["Field", "Value"]),
            [m["crop"] for m in page["marks"]], [img_out, json_out, md_out, xlsx_out, html_out])

import gradio as gr
print("pipeline ready - now run A5")

In [ ]:
# A5. OLD Gradio app. Not needed: use W1 below for the new website.
RUN_OLD_APP = False   # set to True only if you really want the old app
if RUN_OLD_APP:
    import gradio as gr

    with gr.Blocks(title="AksharDrishti") as demo:
        gr.Markdown("# AksharDrishti\nDigitize and verify multilingual Indian government documents. "
                    "OCR engine: IndicOCR (Bodhan AI) · Stamp/signature detection, field extraction, masking, archive: this project.")
        with gr.Tab("Process document"):
            with gr.Row():
                with gr.Column(scale=1):
                    inp = gr.Image(type="filepath", label="Scanned page (JPG/PNG)")
                    mask_cb = gr.Checkbox(value=True, label="Mask Aadhaar / PAN numbers")
                    btn = gr.Button("Process", variant="primary")
                    verdict = gr.Textbox(label="Verification", lines=3)
                with gr.Column(scale=2):
                    ann = gr.Image(label="Layout (blue/green), signatures (red), stamps (violet)")
            with gr.Tab("Extracted text"):
                text_md = gr.Markdown()
            with gr.Tab("Key fields"):
                fields_df = gr.Dataframe(headers=["Field", "Value"])
            with gr.Tab("Stamps & signatures"):
                crops = gr.Gallery(columns=4, height=300)
            with gr.Tab("Downloads"):
                files_out = gr.File(file_count="multiple")
            btn.click(process_document, [inp, mask_cb], [verdict, ann, text_md, fields_df, crops, files_out])
        with gr.Tab("Search archive"):
            q = gr.Textbox(label="Search all processed documents (Hindi / Marathi / English)")
            sbtn = gr.Button("Search")
            res = gr.Dataframe()
            sbtn.click(archive_search, q, res)
            q.submit(archive_search, q, res)

    demo.launch(share=True, server_port=7860, prevent_thread_lock=True)
    if not getattr(demo, "share_url", None):
        # No public link: serve through Colab's proxy. root_path makes image/file links use the proxy
        # address instead of https://localhost:7860, which the browser cannot reach.
        demo.close()
        from google.colab.output import eval_js
        url = eval_js("google.colab.kernel.proxyPort(7860)")
        demo.launch(share=False, server_port=7860, root_path=url.rstrip("/"), prevent_thread_lock=True)
        print("Public link unavailable. Open the app here (works in your browser only):", url)
    # to stop the app: demo.close()


## Akshar Drishti website (opens in its own browser tab)
Run after cells 1-4, 4b, S1, S2, A2, A3 and A4 (A5 is not needed). If asked, upload `webapp/akshar_web.py` and `webapp/akshar_web.html` from this repository. The cell prints a public link; open it in Chrome.

In [ ]:
# W1. Akshar Drishti web app: opens at its own link (run after cells 3, 4, 4b, S1, S2, A2, A3, A4)
# The two website files are downloaded from GitHub automatically; nothing to upload.
import os, urllib.request
BASE = "https://raw.githubusercontent.com/PJTarde/akshar-drishti/main/webapp/"
for f in ("akshar_web.py", "akshar_web.html"):
    urllib.request.urlretrieve(BASE + f, "/content/" + f)
print("website files downloaded")
%run -i /content/akshar_web.py

# Evaluation: finding the numbers for the paper

1. Put your evaluation pages (JPG/PNG) in Google Drive, folder **`akshardrishti_eval/pages`**.
2. Run cells 1-5, S1, S2, A1-A4 (A5 not needed), then **E1**.
3. E1 writes sheets into `akshardrishti_eval` on your Drive. Open them, fill the empty columns, save as CSV (same name).
4. Run **E2**. It prints every number for the paper, ready to paste.

In [ ]:
# E1. Process every evaluation page and create the sheets you fill in
import glob, csv, shutil, time, statistics
from google.colab import drive
drive.mount("/content/drive")

EVAL_DIR = "/content/drive/MyDrive/akshardrishti_eval"
PAGES = sorted(p for p in glob.glob(f"{EVAL_DIR}/pages/*") if p.lower().endswith((".png", ".jpg", ".jpeg")))
print(len(PAGES), "pages found in", f"{EVAL_DIR}/pages")

def write_csv(name, header, rows, keep_existing=True):
    path = f"{EVAL_DIR}/{name}"
    if keep_existing and os.path.exists(path):
        print("kept your existing", name, "(delete it to regenerate)")
        return
    with open(path, "w", encoding="utf-8-sig", newline="") as f:
        w = csv.writer(f); w.writerow(header); w.writerows(rows)

torch.cuda.reset_peak_memory_stats()
marks_rows, field_rows, page_rows, time_rows = [], [], [], []
for i, p in enumerate(PAGES, 1):
    name = os.path.splitext(os.path.basename(p))[0]
    process_document(p, mask=False)          # unmasked, so you can check fields against the page
    t = LAST_TIMING
    time_rows.append([name, round(t["parser"], 2), round(t["marks"], 2), round(t["rest"], 2),
                      round(t["parser"] + t["marks"] + t["rest"], 2)])
    page = json.load(open(f"/content/app_output/{name}/{name}.json", encoding="utf-8"))
    for k, m in enumerate(page["marks"]):
        marks_rows.append([name, k, m["type"], m["method"], m["conf"], os.path.basename(m["crop"]), ""])
    for f_ in page["fields"]:
        field_rows.append([name, f_["field"], f_["value"]])
    page_rows.append([name, "", "", "", "", ""])
    shutil.copytree(f"/content/app_output/{name}", f"{EVAL_DIR}/review/{name}", dirs_exist_ok=True)
    print(f"[{i}/{len(PAGES)}] {name}: {len(page['marks'])} marks, {len(page['fields'])} fields, {time_rows[-1][-1]} s")

peak = torch.cuda.max_memory_allocated() / 1e9
json.dump({"peak_gpu_gb": round(peak, 2)}, open(f"{EVAL_DIR}/summary.json", "w"))
write_csv("timing.csv", ["page", "parser_s", "marks_s", "rest_s", "total_s"], time_rows, keep_existing=False)
write_csv("fields_pred.csv", ["page", "field", "value"], field_rows, keep_existing=False)
write_csv("marks_to_check.csv", ["page", "mark_no", "type", "method", "conf", "crop", "correct (1 or 0)"], marks_rows)
write_csv("fields_true.csv", ["page", "field", "value"], field_rows)
write_csv("pages.csv", ["page", "language (hi/mr/en/mixed)", "scan_type (scan/photo)",
                        "missed_signatures", "missed_colour_stamps", "missed_black_stamps"], page_rows)
print(f"\nPeak GPU memory {peak:.2f} GB. Now fill the sheets in {EVAL_DIR} (see the instructions), then run E2.")

In [ ]:
# E2. Compute every number for the paper from your filled sheets
import pandas as pd, statistics

def rd(n):
    return pd.read_csv(f"{EVAL_DIR}/{n}", encoding="utf-8-sig", dtype=str).fillna("")
def num(x):
    try: return int(float(x))
    except: return 0
def prf(tp, n_pred, n_true):
    P = tp / n_pred if n_pred else float("nan")
    R = tp / n_true if n_true else float("nan")
    F = 2 * P * R / (P + R) if P == P and R == R and (P + R) else float("nan")
    return P, R, F
fmt = lambda x: "--" if x != x else f"{x:.2f}"

pages = rd("pages.csv"); pages.columns = ["page", "lang", "scan", "miss_sig", "miss_col", "miss_blk"]
marks = rd("marks_to_check.csv"); marks.columns = ["page", "mark_no", "type", "method", "conf", "crop", "correct"]
ftrue, fpred = rd("fields_true.csv"), rd("fields_pred.csv")
tim = pd.read_csv(f"{EVAL_DIR}/timing.csv")
peak = json.load(open(f"{EVAL_DIR}/summary.json"))["peak_gpu_gb"]
if (marks["correct"] == "").any():
    print("WARNING:", (marks["correct"] == "").sum(), "marks have no 'correct' value yet - fill them first")
marks["ok"] = marks["correct"].map(num)
out = []
def say(s=""):
    print(s); out.append(s)

# ---- Table 2: dataset --------------------------------------------------------
sig, stp = marks[marks.type == "signature"], marks[marks.type == "stamp"]
tp_sig, tp_st = sig.ok.sum(), stp.ok.sum()
miss_sig = pages.miss_sig.map(num).sum(); miss_col = pages.miss_col.map(num).sum(); miss_blk = pages.miss_blk.map(num).sum()
n_sig, n_col = tp_sig + miss_sig, tp_st + miss_col
lang = pages.lang.str.lower().value_counts(); scan = pages.scan.str.lower().value_counts()
idn = ftrue[ftrue.field.isin(["Aadhaar", "PAN"])]
say("=== Table 2 (dataset) ===")
say(f"Pages: {len(pages)}  (hi/mr/en/mixed = {lang.get('hi',0)} / {lang.get('mr',0)} / {lang.get('en',0)} / {lang.get('mixed',0)})")
say(f"Clean scans / phone photographs = {scan.get('scan',0)} / {scan.get('photo',0)}")
say(f"Annotated signatures = {n_sig}; stamps coloured / black = {n_col} / {miss_blk}")
say(f"Annotated key-field occurrences = {len(ftrue)}; pages with Aadhaar or PAN = {idn.page.nunique()}")

# ---- Table 4: detection ------------------------------------------------------
say("\n=== Table 4 (detection) - paste into LaTeX ===")
rows = [("Signature", n_sig, prf(tp_sig, len(sig), n_sig)),
        ("Stamp (coloured ink)", n_col, prf(tp_st, len(stp), n_col)),
        ("Stamp (black ink)", miss_blk, (float("nan"), 0.0 if miss_blk else float("nan"), float("nan"))),
        ("Stamp (all)", n_col + miss_blk, prf(tp_st, len(stp), n_col + miss_blk))]
for lbl, n, (P, R, F) in rows:
    say(f"{lbl} & {n} & {fmt(P)} & {fmt(R)} & {fmt(F)} \\\\")
# page-level verdict accuracy
per = pages.set_index("page")
ok_s = ok_t = 0
for pg in pages.page:
    m = marks[marks.page == pg]
    true_s = m[(m.type == "signature")].ok.sum() + num(per.loc[pg, "miss_sig"]) > 0
    true_t = m[(m.type == "stamp")].ok.sum() + num(per.loc[pg, "miss_col"]) + num(per.loc[pg, "miss_blk"]) > 0
    ok_s += (len(m[m.type == "signature"]) > 0) == true_s
    ok_t += (len(m[m.type == "stamp"]) > 0) == true_t
say(f"Verdict 'signed' correct on {100*ok_s/len(pages):.1f}% of pages, 'stamped' on {100*ok_t/len(pages):.1f}%")
tagged = 0
for pg in pages.page:
    jp = f"{EVAL_DIR}/review/{pg}/{pg}.json"
    if os.path.exists(jp):
        tagged += sum("inside_mark" in b for b in json.load(open(jp, encoding="utf-8"))["blocks"])
say(f"Parser blocks tagged as lying inside a mark: {tagged}")

# ---- Table 5: key fields -----------------------------------------------------
def norm(v):
    return re.sub(r"\s", "", str(v).translate(DEV_DIGITS)).upper()
def keyset(df):
    return {(r.page, r.field, norm(r.value)) for r in df.itertuples()}
T, Pd = keyset(ftrue), keyset(fpred)
say("\n=== Table 5 (key fields) - paste into LaTeX ===")
tot_tp = tot_p = tot_t = 0
for fld in FIELD_PATTERNS:
    t_ = {x for x in T if x[1] == fld}; p_ = {x for x in Pd if x[1] == fld}
    tp = len(t_ & p_); tot_tp += tp; tot_p += len(p_); tot_t += len(t_)
    P, R, F = prf(tp, len(p_), len(t_))
    say(f"{fld} & {len(t_)} & {fmt(P)} & {fmt(R)} & {fmt(F)} \\\\")
P, R, F = prf(tot_tp, tot_p, tot_t)
say(f"Micro-average & {tot_t} & {fmt(P)} & {fmt(R)} & {fmt(F)} \\\\")

# ---- Masking -----------------------------------------------------------------
say("\n=== Masking ===")
hidden, check_imgs = 0, []
for r in idn.itertuples():
    page = json.load(open(f"{EVAL_DIR}/review/{r.page}/{r.page}.json", encoding="utf-8"))
    alltext = "\n".join([b.get("text") or "" for b in page["blocks"]] + [page.get("markdown", "")])
    masked = mask_text(alltext)
    if norm(r.value) not in norm(masked):
        hidden += 1
    # image check: draw the masked regions so you can look at them
    src = f"{EVAL_DIR}/review/{r.page}/{r.page}_input.png"
    if os.path.exists(src) and r.page not in check_imgs:
        im = Image.open(src).convert("RGB"); d = ImageDraw.Draw(im)
        for hb in sensitive_boxes(page):
            d.rectangle(hb, fill="black")
        im.save(f"{EVAL_DIR}/review/{r.page}/masked_check.png"); check_imgs.append(r.page)
say(f"Text masking: {hidden} of {len(idn)} identity numbers hidden ({100*hidden/max(len(idn),1):.1f}%)")
say(f"Image masking: open review/<page>/masked_check.png for these {len(check_imgs)} pages and count how many numbers are fully hidden: {check_imgs}")

# ---- Table 6: time -----------------------------------------------------------
say("\n=== Table 6 (time per page, seconds) - paste into LaTeX ===")
for lbl, col in [("Parser (layout + recognition)", "parser_s"), ("Signature + stamp detection", "marks_s"),
                 ("Fields, masking, outputs, archive", "rest_s"), ("Total", "total_s")]:
    say(f"{lbl} & {tim[col].mean():.2f} & {tim[col].median():.2f} \\\\")
say(f"Peak GPU memory: {peak} GB")

open(f"{EVAL_DIR}/results_for_paper.txt", "w", encoding="utf-8").write("\n".join(out))
print(f"\nSaved to {EVAL_DIR}/results_for_paper.txt")